# 02. Preprocessing Policy

이 노트북은 OTT 이탈 예측 프로젝트의 **전처리 정책을 코드로 고정하는 단계**이다.

01번 노트북이 원본 데이터의 구조와 key 관계를 확인하는 단계였다면, 02번 노트북은 이후 모든 분석에서 공통으로 사용할 분석 대상과 관측창을 확정한다.

이 노트북의 핵심 질문은 다음과 같다.

1. 어떤 고객/구독 행을 분석 대상에 포함할 것인가?
2. 어떤 행을 더미 이상치 또는 관측 불가능 케이스로 제외할 것인가?
3. 100원딜 고객을 어떻게 정의할 것인가?
4. `USER_KEY`와 `USER_NUM`의 연결 관계를 어떻게 처리할 것인가?
5. 고객별 1~3주차 관측창을 어떻게 정의할 것인가?
6. 이후 노트북들이 안정적으로 사용할 중간 산출물을 어떤 파일로 저장할 것인가?

중요한 원칙은 하나다. **이 노트북은 모델링을 하지 않는다.**  
이 노트북은 분석에 들어가기 전에 재료를 세척하고, 같은 크기로 손질하고, 어떤 재료를 쓸지 정하는 단계다.

## 2-1. 이 노트북의 입력과 출력

### 입력

원본 데이터는 `park.ingyeom/_data/01_raw/` 아래에 있다고 가정한다. 다만 실행 환경에 따라 `_data/raw`, `_data`, `data/raw`, `data`에 놓인 경우도 찾을 수 있도록 경로 탐색 함수를 포함한다.

필수 입력 파일은 다음과 같다.

| 파일 | 용도 |
|---|---|
| `Membership_v1.csv` | target인 `is_repurchase`와 멤버십/요금제/프로모션 정보 |
| `User_Mapping_v1.csv` | `USER_KEY`와 `USER_NUM` 연결 |
| `View_History_v1.csv` | 고객별 시청이력, `watch_day`, `watch_time`, `MOVIE_NUM` |

### 출력

이 노트북은 다음 파일을 생성한다.

| 출력 파일 | 설명 |
|---|---|
| `_data/02_interim/membership_preprocessed.csv` | 더미 이상치와 짧은 구독기간을 제외하고 기본 flag를 추가한 멤버십 테이블 |
| `_data/02_interim/membership_with_usernum.csv` | 전처리된 멤버십에 `USER_NUM`을 연결한 테이블 |
| `_data/02_interim/view_history_observation_window.csv` | 고객별 `reg_date` 기준 day 0~20 관측창 안의 시청이력 |
| `_data/02_interim/preprocessing_summary.json` | 전처리 요약 수치 |
| `reports/tables/02_preprocessing_*.csv` | 검산용 표 |

## 2-2. 전처리 정책 요약

이 노트북에서 적용하는 정책은 다음과 같다.

| 정책 | 결정 |
|---|---|
| 분석 단위 | 개인 생애 단위가 아니라 **구독 이벤트 또는 Membership 행 단위** |
| 더미 이상치 | `gender == 'N'`, `is_user_verified == 0`, `age == 40` 동시 만족 행 제거 |
| 구독기간 | 고객별 3주 관측창을 만들기 위해 `subscription_days >= 21`만 유지 |
| 100원딜 정의 | `price == 100`을 `is_100won`으로 정의하고, `is_promotion`과의 일치 여부 검산 |
| `USER_KEY` 중복 | 오류로 단정하지 않음. 복수 계정 또는 재가입 가능성으로 보고 기록 후 유지 |
| 관측창 | 고객별 `reg_date` 기준 day 0~20, 즉 가입 후 1~3주차 |
| 4주차 | 피처가 아니라 리텐션 대응기간. 이 노트북에서는 포함하지 않음 |
| 시청이력 없음 | 삭제하지 않고 이후 feature 단계에서 `no_watch_obs_flag`로 표현할 수 있도록 보존 |

주의할 점은 `USER_KEY`와 `USER_NUM`의 관계다. `User_Mapping`에는 `USER_KEY` 중복이 일부 존재한다. 이 프로젝트에서는 이를 무조건 데이터 오류로 보지 않는다. 동일 개인의 복수 계정 생성, 구독 취소 후 다른 계정 재가입 등으로 생긴 자연스러운 현상일 수 있기 때문이다.

따라서 이 노트북에서는 `membership_row_id`를 새로 만들어 각 멤버십 행을 고유하게 추적한다. 이후 파생변수 생성도 가능하면 `membership_row_id` 기준으로 수행하는 것이 안전하다.

In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2-3. 경로 설정

GitHub 저장소에서는 `park.ingyeom` 폴더 안에서 작업한다고 가정한다.  
노트북을 어느 위치에서 실행하더라도 `_data` 또는 `data` 폴더를 상위 경로에서 찾도록 구성한다.

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    """
    Find project root by walking upward until a directory containing
    `_data`, `data`, `notebooks`, or `src` is found.

    Recommended execution location:
    park.ingyeom/notebooks/02_preprocessing/02_preprocessing_policy.ipynb
    """
    if start is None:
        start = Path.cwd()
    start = start.resolve()

    candidates = [start, *start.parents]
    for path in candidates:
        if (path / "_data").exists() or (path / "data").exists():
            return path
        if (path / "notebooks").exists() and (path / "src").exists():
            return path

    # Fallback: current working directory
    return start


PROJECT_ROOT = find_project_root()

DATA_ROOT_CANDIDATES = [
    PROJECT_ROOT / "_data",
    PROJECT_ROOT / "data",
    Path("/mnt/data"),  # fallback for ChatGPT execution environment
]

DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])

RAW_DIR_CANDIDATES = [
    DATA_ROOT / "01_raw",
    DATA_ROOT / "raw",
    DATA_ROOT,
]
RAW_DIR = next((p for p in RAW_DIR_CANDIDATES if p.exists()), RAW_DIR_CANDIDATES[0])

INTERIM_DIR = DATA_ROOT / "02_interim"
PROCESSED_DIR = DATA_ROOT / "03_processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"

for d in [INTERIM_DIR, PROCESSED_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("RAW_DIR:", RAW_DIR)
print("INTERIM_DIR:", INTERIM_DIR)
print("TABLES_DIR:", TABLES_DIR)

PROJECT_ROOT: /mnt/data/test_repo/park.ingyeom
DATA_ROOT: /mnt/data/test_repo/park.ingyeom/_data
RAW_DIR: /mnt/data/test_repo/park.ingyeom/_data/01_raw
INTERIM_DIR: /mnt/data/test_repo/park.ingyeom/_data/02_interim
TABLES_DIR: /mnt/data/test_repo/park.ingyeom/reports/tables


In [3]:
FILE_NAMES = {
    "membership": "Membership_v1.csv",
    "mapping": "User_Mapping_v1.csv",
    "view": "View_History_v1.csv",
}


def find_file(file_name: str, search_dirs: list[Path]) -> Path:
    """Find a file in common raw-data locations."""
    for directory in search_dirs:
        candidate = directory / file_name
        if candidate.exists():
            return candidate

    # Recursive fallback, limited to the current data root and /mnt/data.
    for base in [DATA_ROOT, Path("/mnt/data")]:
        if base.exists():
            matches = list(base.rglob(file_name))
            if matches:
                return matches[0]

    raise FileNotFoundError(f"Could not find {file_name}. Put it under _data/01_raw or update FILE_NAMES.")


PATHS = {key: find_file(name, [RAW_DIR, DATA_ROOT, Path("/mnt/data")]) for key, name in FILE_NAMES.items()}
PATHS

{'membership': PosixPath('/mnt/data/test_repo/park.ingyeom/_data/01_raw/Membership_v1.csv'),
 'mapping': PosixPath('/mnt/data/test_repo/park.ingyeom/_data/01_raw/User_Mapping_v1.csv'),
 'view': PosixPath('/mnt/data/test_repo/park.ingyeom/_data/01_raw/View_History_v1.csv')}

## 2-4. 데이터 로딩

이 단계에서는 원본 CSV를 읽고, 컬럼명을 후속 코드에서 쓰기 편하게 최소한으로 정리한다.

`View_History_v1.csv`의 시청시간 컬럼은 원본에서 `watch_time(min)` 형태이므로, 이후 코드에서는 `watch_time`으로 통일한다.

In [4]:
membership_raw = pd.read_csv(PATHS["membership"])
mapping_raw = pd.read_csv(PATHS["mapping"])
view_raw = pd.read_csv(PATHS["view"])

view_raw = view_raw.rename(columns={"watch_time(min)": "watch_time"})

file_summary = pd.DataFrame([
    {"name": "membership_raw", "path": str(PATHS["membership"]), "rows": len(membership_raw), "cols": membership_raw.shape[1]},
    {"name": "mapping_raw", "path": str(PATHS["mapping"]), "rows": len(mapping_raw), "cols": mapping_raw.shape[1]},
    {"name": "view_raw", "path": str(PATHS["view"]), "rows": len(view_raw), "cols": view_raw.shape[1]},
])

display(file_summary)
file_summary.to_csv(TABLES_DIR / "02_preprocessing_file_summary.csv", index=False, encoding="utf-8-sig")

,name,path,rows,cols
0,membership_raw,/mnt/data/test_repo/park.ingyeom/_data/01_raw/...,17876,15
1,mapping_raw,/mnt/data/test_repo/park.ingyeom/_data/01_raw/...,19877,2
2,view_raw,/mnt/data/test_repo/park.ingyeom/_data/01_raw/...,106205,5


In [5]:
print("Membership columns")
print(membership_raw.columns.tolist())
print("\nMapping columns")
print(mapping_raw.columns.tolist())
print("\nView columns")
print(view_raw.columns.tolist())

Membership columns
['USER_KEY', 'product_code', 'price', 'billing_method', 'max_screen', 'is_promotion', 'is_churn_prevented', 'is_repurchase', 'payment_device', 'is_user_verified', 'gender', 'age', 'reg_date', 'reg_hour', 'end_date']

Mapping columns
['USER_KEY', 'USER_NUM']

View columns
['USER_NUM', 'MOVIE_NUM', 'watch_time', 'watch_day', 'watch_seq']


## 2-5. 날짜와 기본 타입 정리

`reg_date`, `end_date`, `watch_day`를 날짜형으로 변환한다.  
`watch_day`는 원본에서 `20210314`처럼 정수형 또는 문자열형으로 들어올 수 있으므로 `format='%Y%m%d'`를 우선 사용한다.

In [6]:
def parse_yyyymmdd(series: pd.Series) -> pd.Series:
    """Parse yyyymmdd-like values robustly."""
    s = series.astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    return pd.to_datetime(s, format="%Y%m%d", errors="coerce")

membership = membership_raw.copy()
mapping = mapping_raw.copy()
view = view_raw.copy()

membership["reg_date"] = pd.to_datetime(membership["reg_date"], errors="coerce")
membership["end_date"] = pd.to_datetime(membership["end_date"], errors="coerce")
view["watch_day"] = parse_yyyymmdd(view["watch_day"])

# Basic numeric coercion for columns used in preprocessing.
for col in ["price", "max_screen", "is_promotion", "is_churn_prevented", "is_repurchase", "is_user_verified", "age"]:
    if col in membership.columns:
        membership[col] = pd.to_numeric(membership[col], errors="coerce")

for col in ["USER_NUM", "MOVIE_NUM", "watch_time", "watch_seq"]:
    if col in view.columns:
        view[col] = pd.to_numeric(view[col], errors="coerce")

print("membership reg_date range:", membership["reg_date"].min(), "~", membership["reg_date"].max())
print("membership end_date range:", membership["end_date"].min(), "~", membership["end_date"].max())
print("view watch_day range:", view["watch_day"].min(), "~", view["watch_day"].max())

membership reg_date range: 2021-03-01 00:00:00 ~ 2021-03-15 00:00:00
membership end_date range: 2021-03-01 00:00:00 ~ 2021-04-16 00:00:00
view watch_day range: 2021-03-01 00:00:00 ~ 2021-04-05 00:00:00


## 2-6. 멤버십 행 고유 ID 부여

분석 단위는 개인 생애 단위가 아니라 멤버십 행 또는 구독 이벤트 단위다.  
따라서 원본 `Membership`의 각 행에 `membership_row_id`를 부여한다.

이 값은 이후 `USER_KEY`가 중복 매핑되더라도 원래 멤버십 행을 추적하기 위한 기준점이다.

In [7]:
membership = membership.reset_index(drop=True).copy()
membership.insert(0, "membership_row_id", np.arange(len(membership), dtype=int))

membership[["membership_row_id", "USER_KEY", "price", "is_promotion", "is_repurchase", "reg_date", "end_date"]].head()

,membership_row_id,USER_KEY,price,is_promotion,is_repurchase,reg_date,end_date
0,0,7a6960912bebe03c6e4c770eb1aa91329c3497f18f90ca...,100.0000,1,0,2021-03-14,2021-04-14
1,1,4ec765db76545c1d6dda9f421590bf9d02f584009f8d92...,100.0000,1,0,2021-03-09,2021-04-09
2,2,4f86d917c53cb6bd8949f76dba7260311e8c1748748a02...,100.0000,1,0,2021-03-09,2021-04-09
3,3,445fb8813626d3d49b94b5be58cd76d80ed31fa94f8372...,9.9900,0,1,2021-03-09,2021-04-10
4,4,01b16f9f7ff29b48b1ee0d1a89d1eb9662474e5eedb8c2...,100.0000,1,1,2021-03-09,2021-04-09


## 2-7. 구독기간 계산과 21일 미만 케이스 확인

멘토의 문제 정의는 **가입 후 1~3주차 행동을 관측하고, 4주차를 리텐션 대응기간으로 남긴다**는 구조다.  
따라서 최소한 21일의 관측 가능 기간이 필요하다.

이 노트북에서는 `subscription_days = end_date - reg_date + 1`로 계산한다.  
`subscription_days < 21`인 행은 3주 관측창을 안정적으로 만들기 어렵기 때문에 제외한다.

In [8]:
membership["subscription_days"] = (membership["end_date"] - membership["reg_date"]).dt.days + 1

subscription_summary = membership["subscription_days"].describe().to_frame("subscription_days")
subscription_short_count = int((membership["subscription_days"] < 21).sum())
subscription_missing_count = int(membership["subscription_days"].isna().sum())

print("subscription_days < 21:", subscription_short_count)
print("subscription_days missing:", subscription_missing_count)
display(subscription_summary)

subscription_summary.to_csv(TABLES_DIR / "02_preprocessing_subscription_days_summary.csv", encoding="utf-8-sig")

subscription_days < 21: 455
subscription_days missing: 0


,subscription_days
count,"17,876.0000"
mean,31.3938
std,4.7922
min,1.0000
25%,32.0000
50%,32.0000
75%,32.0000
max,33.0000


## 2-8. 더미 이상치 제거 정책

원본 브리핑에서 확인된 핵심 이상치는 다음 조건이다.

```text
(gender == 'N') AND (is_user_verified == 0) AND (age == 40)
```

이 조합은 실제 40대 미인증 고객이라기보다, 본인인증을 하지 않은 고객에게 성별/연령 기본값이 채워진 더미 데이터로 판단한다.  
따라서 분석 대상에서 제외한다.

In [9]:
dummy_mask = (
    (membership["gender"] == "N")
    & (membership["is_user_verified"] == 0)
    & (membership["age"] == 40)
)

anomaly_summary = pd.DataFrame([
    {"item": "membership_rows_before", "value": len(membership)},
    {"item": "dummy_anomaly_rows", "value": int(dummy_mask.sum())},
    {"item": "dummy_anomaly_rate", "value": float(dummy_mask.mean())},
])

display(anomaly_summary)

anomaly_by_promo = (
    membership.assign(dummy_anomaly=dummy_mask.astype(int))
    .groupby(["is_promotion", "dummy_anomaly"], dropna=False)["membership_row_id"]
    .count()
    .reset_index(name="n")
)

display(anomaly_by_promo)

anomaly_summary.to_csv(TABLES_DIR / "02_preprocessing_dummy_anomaly_summary.csv", index=False, encoding="utf-8-sig")
anomaly_by_promo.to_csv(TABLES_DIR / "02_preprocessing_dummy_anomaly_by_promotion.csv", index=False, encoding="utf-8-sig")

,item,value
0,membership_rows_before,"17,876.0000"
1,dummy_anomaly_rows,"2,639.0000"
2,dummy_anomaly_rate,0.1476


,is_promotion,dummy_anomaly,n
0,0,0,6176
1,0,1,2639
2,1,0,9061


## 2-9. 100원딜 정의와 `is_promotion` 관계 확인

현재 프로젝트에서는 `price == 100`을 100원딜 고객으로 정의한다.  
다만 `is_promotion == 1`과 완전히 같은지 반드시 검산한다.

이 검산은 매우 중요하다. 두 컬럼이 완전히 일치하면 이후 분석에서 `is_100won`과 `is_promotion`을 같은 축으로 볼 수 있다.  
일치하지 않는다면 `100원딜`, `기타 프로모션`, `일반 고객`을 분리해야 한다.

In [10]:
membership["is_100won"] = (membership["price"] == 100).astype(int)

promo_price_crosstab = pd.crosstab(
    membership["is_promotion"],
    membership["is_100won"],
    rownames=["is_promotion"],
    colnames=["is_100won"],
    dropna=False,
)

display(promo_price_crosstab)

mismatch_mask = membership["is_promotion"].fillna(-1).astype(int) != membership["is_100won"].fillna(-1).astype(int)
print("is_promotion vs is_100won mismatch rows:", int(mismatch_mask.sum()))

promo_price_crosstab.to_csv(TABLES_DIR / "02_preprocessing_promotion_price_crosstab.csv", encoding="utf-8-sig")

is_100won,0,1
is_promotion,,
0,8815,0
1,0,9061


is_promotion vs is_100won mismatch rows: 0


## 2-10. 기본 flag와 연령대 변수 생성

전처리된 멤버십 테이블에는 이후 분석에서 반복적으로 사용할 기본 flag를 추가한다.

생성 변수는 다음과 같다.

| 변수 | 의미 |
|---|---|
| `is_100won` | `price == 100` 여부 |
| `screen_1_flag` | 1인 요금제 여부 |
| `screen_2_flag` | 2인 요금제 여부 |
| `screen_4_flag` | 4인 요금제 여부 |
| `promo_x_1screen` | 100원딜이면서 1인 요금제 |
| `promo_x_2screen` | 100원딜이면서 2인 요금제 |
| `promo_x_4screen` | 100원딜이면서 4인 요금제 |
| `age_band` | 연령대 |

In [11]:
def add_membership_flags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["is_100won"] = (out["price"] == 100).astype(int)

    out["screen_1_flag"] = (out["max_screen"] == 1).astype(int)
    out["screen_2_flag"] = (out["max_screen"] == 2).astype(int)
    out["screen_4_flag"] = (out["max_screen"] == 4).astype(int)

    out["promo_x_1screen"] = ((out["is_100won"] == 1) & (out["max_screen"] == 1)).astype(int)
    out["promo_x_2screen"] = ((out["is_100won"] == 1) & (out["max_screen"] == 2)).astype(int)
    out["promo_x_4screen"] = ((out["is_100won"] == 1) & (out["max_screen"] == 4)).astype(int)

    bins = [0, 19, 29, 39, 49, 59, 120]
    labels = ["10s_or_less", "20s", "30s", "40s", "50s", "60s_plus"]
    out["age_band"] = pd.cut(out["age"], bins=bins, labels=labels, right=True)

    return out


membership_with_flags = add_membership_flags(membership)

flag_check = membership_with_flags.groupby(["is_100won", "max_screen"], dropna=False)["is_repurchase"].agg(
    n="count",
    repurchase_rate="mean",
).reset_index()

display(flag_check)
flag_check.to_csv(TABLES_DIR / "02_preprocessing_100won_maxscreen_check_before_filter.csv", index=False, encoding="utf-8-sig")

,is_100won,max_screen,n,repurchase_rate
0,0,1,5745,0.6754
1,0,2,2225,0.7443
2,0,4,845,0.7337
3,1,1,5484,0.6495
4,1,2,1532,0.7383
5,1,4,2045,0.4802


## 2-11. 멤버십 전처리 적용

이제 실제 분석용 멤버십 테이블을 만든다.

적용 순서는 다음과 같다.

1. 원본 멤버십 행에 `membership_row_id` 부여
2. 날짜와 기본 타입 정리
3. `subscription_days` 계산
4. 더미 이상치 제거
5. `subscription_days >= 21`만 유지
6. 100원딜, 요금제, 연령대 flag 생성

이 결과물이 이후 모든 노트북의 기본 멤버십 테이블이다.

In [12]:
rows_before = len(membership_with_flags)

valid_mask = (
    (~dummy_mask)
    & (membership_with_flags["subscription_days"] >= 21)
)

membership_preprocessed = membership_with_flags.loc[valid_mask].copy().reset_index(drop=True)

rows_after = len(membership_preprocessed)

membership_preprocess_summary = pd.DataFrame([
    {"step": "raw_membership", "rows": rows_before},
    {"step": "remove_dummy_anomaly", "rows_removed": int(dummy_mask.sum())},
    {"step": "remove_subscription_days_lt_21", "rows_removed": int(((~dummy_mask) & (membership_with_flags["subscription_days"] < 21)).sum())},
    {"step": "membership_preprocessed", "rows": rows_after},
])

display(membership_preprocess_summary)

repurchase_by_100won = membership_preprocessed.groupby("is_100won")["is_repurchase"].agg(
    n="count",
    repurchase_rate="mean",
).reset_index()

display(repurchase_by_100won)

repurchase_by_screen = membership_preprocessed.groupby(["is_100won", "max_screen"], dropna=False)["is_repurchase"].agg(
    n="count",
    repurchase_rate="mean",
).reset_index()

display(repurchase_by_screen)

membership_preprocess_summary.to_csv(TABLES_DIR / "02_preprocessing_membership_summary.csv", index=False, encoding="utf-8-sig")
repurchase_by_100won.to_csv(TABLES_DIR / "02_preprocessing_repurchase_by_100won.csv", index=False, encoding="utf-8-sig")
repurchase_by_screen.to_csv(TABLES_DIR / "02_preprocessing_repurchase_by_100won_maxscreen.csv", index=False, encoding="utf-8-sig")

,step,rows,rows_removed
0,raw_membership,"17,876.0000",NaN
1,remove_dummy_anomaly,NaN,"2,639.0000"
2,remove_subscription_days_lt_21,NaN,315.0000
3,membership_preprocessed,"14,922.0000",NaN


,is_100won,n,repurchase_rate
0,0,5939,0.7410
1,1,8983,0.6317


,is_100won,max_screen,n,repurchase_rate
0,0,1,3794,0.7193
1,0,2,1594,0.7842
2,0,4,551,0.7659
3,1,1,5413,0.6580
4,1,2,1528,0.7402
5,1,4,2042,0.4809


## 2-12. `USER_KEY`와 `USER_NUM` 연결 정책

`Membership`은 `USER_KEY`를 기준으로 하고, `View_History`는 `USER_NUM`을 기준으로 한다.  
따라서 두 데이터를 연결하려면 `User_Mapping`을 거쳐야 한다.

이 프로젝트에서는 `USER_KEY` 중복을 무조건 제거하지 않는다.  
중복은 다음 상황에서 생길 수 있다.

1. 한 개인이 복수 계정을 만든 경우
2. 기존 계정을 취소하고 다른 계정으로 다시 가입한 경우
3. 동일 고객의 여러 이용 식별자가 남아 있는 경우

따라서 여기서는 중복을 **기록하고 유지**한다.  
대신 `membership_row_id`를 기준으로 원래 멤버십 행을 추적한다.

In [13]:
mapping_check = pd.DataFrame([
    {"item": "mapping_rows", "value": len(mapping)},
    {"item": "unique_USER_KEY", "value": mapping["USER_KEY"].nunique()},
    {"item": "unique_USER_NUM", "value": mapping["USER_NUM"].nunique()},
    {"item": "duplicated_USER_KEY_rows", "value": int(mapping["USER_KEY"].duplicated(keep=False).sum())},
    {"item": "duplicated_USER_KEY_unique", "value": int(mapping.loc[mapping["USER_KEY"].duplicated(keep=False), "USER_KEY"].nunique())},
    {"item": "duplicated_USER_NUM_rows", "value": int(mapping["USER_NUM"].duplicated(keep=False).sum())},
])

display(mapping_check)
mapping_check.to_csv(TABLES_DIR / "02_preprocessing_user_mapping_check.csv", index=False, encoding="utf-8-sig")

,item,value
0,mapping_rows,19877
1,unique_USER_KEY,19828
2,unique_USER_NUM,19877
3,duplicated_USER_KEY_rows,97
4,duplicated_USER_KEY_unique,48
5,duplicated_USER_NUM_rows,0


In [14]:
membership_with_usernum = membership_preprocessed.merge(mapping, on="USER_KEY", how="left", indicator=True)

join_summary = pd.DataFrame([
    {"item": "membership_preprocessed_rows", "value": len(membership_preprocessed)},
    {"item": "membership_with_usernum_rows", "value": len(membership_with_usernum)},
    {"item": "rows_with_USER_NUM", "value": int(membership_with_usernum["USER_NUM"].notna().sum())},
    {"item": "rows_without_USER_NUM", "value": int(membership_with_usernum["USER_NUM"].isna().sum())},
    {"item": "unique_membership_row_id", "value": int(membership_with_usernum["membership_row_id"].nunique())},
    {"item": "membership_rows_with_multiple_USER_NUM", "value": int((membership_with_usernum.groupby("membership_row_id")["USER_NUM"].nunique() > 1).sum())},
])

display(join_summary)
join_summary.to_csv(TABLES_DIR / "02_preprocessing_membership_mapping_join_summary.csv", index=False, encoding="utf-8-sig")

membership_with_usernum = membership_with_usernum.drop(columns=["_merge"])

,item,value
0,membership_preprocessed_rows,14922
1,membership_with_usernum_rows,14955
2,rows_with_USER_NUM,14955
3,rows_without_USER_NUM,0
4,unique_membership_row_id,14922
5,membership_rows_with_multiple_USER_NUM,32


## 2-13. 고객별 3주 관측창 생성

관측창은 전체 날짜 기준이 아니라 **각 고객의 `reg_date` 기준 상대일**로 정의한다.

```text
watch_rel_day = watch_day - reg_date
관측창 = 0 <= watch_rel_day <= 20
```

이 기준은 중요하다. 전체 `watch_day`를 2021-03-01 기준으로 1주차, 2주차, 3주차로 나누면 고객별 가입일 차이를 반영하지 못할 수 있다.  
따라서 여기서는 고객별 상대일 기준을 사용한다.

4주차에 해당하는 day 21~27은 리텐션 대응기간으로 보고 피처에서 제외한다.

In [15]:
# Keep only columns needed to attach observation window.
base_for_view = membership_with_usernum[[
    "membership_row_id", "USER_KEY", "USER_NUM", "reg_date", "end_date", "is_100won", "max_screen", "is_repurchase"
]].copy()

# USER_NUM missing rows cannot be joined to View_History, but they remain in membership tables.
base_for_view_nonmissing = base_for_view.loc[base_for_view["USER_NUM"].notna()].copy()
base_for_view_nonmissing["USER_NUM"] = base_for_view_nonmissing["USER_NUM"].astype(view["USER_NUM"].dropna().dtype)

view_joined = view.merge(base_for_view_nonmissing, on="USER_NUM", how="inner")
view_joined["watch_rel_day"] = (view_joined["watch_day"] - view_joined["reg_date"]).dt.days

view_joined["obs_period_status"] = np.select(
    [
        view_joined["watch_rel_day"] < 0,
        view_joined["watch_rel_day"].between(0, 20, inclusive="both"),
        view_joined["watch_rel_day"].between(21, 27, inclusive="both"),
        view_joined["watch_rel_day"] > 27,
    ],
    ["before_reg_date", "observation_day_0_20", "action_window_day_21_27", "after_action_window"],
    default="missing_or_invalid",
)

obs_view = view_joined.loc[view_joined["obs_period_status"] == "observation_day_0_20"].copy()
obs_view["obs_week"] = pd.cut(
    obs_view["watch_rel_day"],
    bins=[-1, 6, 13, 20],
    labels=[1, 2, 3],
).astype(int)

period_summary = (
    view_joined.groupby("obs_period_status")
    .agg(
        rows=("watch_time", "count"),
        watch_time_sum=("watch_time", "sum"),
        unique_membership_rows=("membership_row_id", "nunique"),
        unique_user_num=("USER_NUM", "nunique"),
    )
    .reset_index()
)

display(period_summary)
period_summary.to_csv(TABLES_DIR / "02_preprocessing_view_period_summary.csv", index=False, encoding="utf-8-sig")

,obs_period_status,rows,watch_time_sum,unique_membership_rows,unique_user_num
0,action_window_day_21_27,3973,178890,2168,2166
1,after_action_window,13,369,3,4
2,before_reg_date,48,2519,14,16
3,observation_day_0_20,85066,3800349,12302,12303


In [16]:
obs_week_summary = (
    obs_view.groupby("obs_week")
    .agg(
        rows=("watch_time", "count"),
        watch_time_sum=("watch_time", "sum"),
        unique_membership_rows=("membership_row_id", "nunique"),
        unique_user_num=("USER_NUM", "nunique"),
        unique_movies=("MOVIE_NUM", "nunique"),
    )
    .reset_index()
)

display(obs_week_summary)
obs_week_summary.to_csv(TABLES_DIR / "02_preprocessing_observation_week_summary.csv", index=False, encoding="utf-8-sig")

,obs_week,rows,watch_time_sum,unique_membership_rows,unique_user_num,unique_movies
0,1,28716,1288791,8373,8376,3066
1,2,28174,1257928,8245,8238,2979
2,3,28176,1253630,8366,8359,3006


## 2-14. 시청이력 없는 고객 처리 정책

이 노트북에서는 시청이력 없는 고객을 삭제하지 않는다.  
시청이력 없음 자체가 의미 있는 행동 정보일 수 있기 때문이다.

다만 실제 시청 행동 파생변수는 04번 노트북에서 만들며, 그때 다음과 같은 flag를 부여할 수 있다.

| 변수 | 의미 |
|---|---|
| `has_watch_obs` | day 0~20 관측창 내 시청이력 있음 |
| `no_watch_obs_flag` | day 0~20 관측창 내 시청이력 없음 |

02번에서는 관측창 안에 시청이력이 있는 멤버십 행 수와 없는 멤버십 행 수만 검산한다.

In [17]:
watched_membership_ids = set(obs_view["membership_row_id"].dropna().unique())
membership_preprocessed["has_watch_obs"] = membership_preprocessed["membership_row_id"].isin(watched_membership_ids).astype(int)
membership_preprocessed["no_watch_obs_flag"] = (membership_preprocessed["has_watch_obs"] == 0).astype(int)

watch_presence_summary = membership_preprocessed.groupby("has_watch_obs")["is_repurchase"].agg(
    n="count",
    repurchase_rate="mean",
).reset_index()
watch_presence_summary["has_watch_obs_label"] = watch_presence_summary["has_watch_obs"].map({0: "no_observation_window_watch", 1: "has_observation_window_watch"})

display(watch_presence_summary[["has_watch_obs_label", "n", "repurchase_rate"]])
watch_presence_summary.to_csv(TABLES_DIR / "02_preprocessing_watch_presence_summary.csv", index=False, encoding="utf-8-sig")

,has_watch_obs_label,n,repurchase_rate
0,no_observation_window_watch,2620,0.6767
1,has_observation_window_watch,12302,0.6749


## 2-15. 최종 전처리 산출물 저장

이 노트북의 핵심 산출물은 세 가지다.

1. `membership_preprocessed.csv`  
   더미 이상치와 21일 미만 구독기간을 제외하고 기본 flag를 추가한 멤버십 테이블

2. `membership_with_usernum.csv`  
   전처리된 멤버십에 `USER_NUM`을 연결한 테이블

3. `view_history_observation_window.csv`  
   고객별 `reg_date` 기준 day 0~20에 해당하는 시청이력

이 파일들은 이후 노트북의 입력으로 사용한다.

In [18]:
# Re-attach has_watch flags to membership_with_usernum as well.
membership_with_usernum = membership_with_usernum.drop(columns=["has_watch_obs", "no_watch_obs_flag"], errors="ignore")
membership_with_usernum = membership_with_usernum.merge(
    membership_preprocessed[["membership_row_id", "has_watch_obs", "no_watch_obs_flag"]],
    on="membership_row_id",
    how="left",
)

PATH_MEMBERSHIP_PREPROCESSED = INTERIM_DIR / "membership_preprocessed.csv"
PATH_MEMBERSHIP_WITH_USERNUM = INTERIM_DIR / "membership_with_usernum.csv"
PATH_OBS_VIEW = INTERIM_DIR / "view_history_observation_window.csv"
PATH_SUMMARY_JSON = INTERIM_DIR / "preprocessing_summary.json"

membership_preprocessed.to_csv(PATH_MEMBERSHIP_PREPROCESSED, index=False, encoding="utf-8-sig")
membership_with_usernum.to_csv(PATH_MEMBERSHIP_WITH_USERNUM, index=False, encoding="utf-8-sig")
obs_view.to_csv(PATH_OBS_VIEW, index=False, encoding="utf-8-sig")

summary = {
    "raw_rows": {
        "membership": int(len(membership_raw)),
        "mapping": int(len(mapping_raw)),
        "view_history": int(len(view_raw)),
    },
    "membership_preprocessing": {
        "rows_before": int(rows_before),
        "dummy_anomaly_rows": int(dummy_mask.sum()),
        "subscription_days_lt_21_removed": int(((~dummy_mask) & (membership_with_flags["subscription_days"] < 21)).sum()),
        "rows_after": int(len(membership_preprocessed)),
        "has_watch_obs_rows": int(membership_preprocessed["has_watch_obs"].sum()),
        "no_watch_obs_rows": int(membership_preprocessed["no_watch_obs_flag"].sum()),
    },
    "mapping": {
        "mapping_rows": int(len(mapping)),
        "unique_USER_KEY": int(mapping["USER_KEY"].nunique()),
        "unique_USER_NUM": int(mapping["USER_NUM"].nunique()),
        "duplicated_USER_KEY_rows": int(mapping["USER_KEY"].duplicated(keep=False).sum()),
        "duplicated_USER_KEY_unique": int(mapping.loc[mapping["USER_KEY"].duplicated(keep=False), "USER_KEY"].nunique()),
        "membership_with_usernum_rows": int(len(membership_with_usernum)),
    },
    "observation_window": {
        "definition": "customer-specific reg_date day 0 to day 20",
        "obs_view_rows": int(len(obs_view)),
        "obs_view_unique_membership_rows": int(obs_view["membership_row_id"].nunique()),
        "obs_view_unique_user_num": int(obs_view["USER_NUM"].nunique()),
        "obs_view_unique_movies": int(obs_view["MOVIE_NUM"].nunique()),
        "obs_view_watch_time_sum": float(obs_view["watch_time"].sum()),
    },
    "outputs": {
        "membership_preprocessed": str(PATH_MEMBERSHIP_PREPROCESSED),
        "membership_with_usernum": str(PATH_MEMBERSHIP_WITH_USERNUM),
        "view_history_observation_window": str(PATH_OBS_VIEW),
        "preprocessing_summary": str(PATH_SUMMARY_JSON),
    },
}

with open(PATH_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved:")
print(PATH_MEMBERSHIP_PREPROCESSED)
print(PATH_MEMBERSHIP_WITH_USERNUM)
print(PATH_OBS_VIEW)
print(PATH_SUMMARY_JSON)

Saved:
/mnt/data/test_repo/park.ingyeom/_data/02_interim/membership_preprocessed.csv
/mnt/data/test_repo/park.ingyeom/_data/02_interim/membership_with_usernum.csv
/mnt/data/test_repo/park.ingyeom/_data/02_interim/view_history_observation_window.csv
/mnt/data/test_repo/park.ingyeom/_data/02_interim/preprocessing_summary.json


## 2-16. 최종 검산

02번 노트북의 마지막 검산이다.

확인할 것:

1. 100원딜 고객과 비100원딜 고객의 재구독률이 기대한 방향으로 나오는가?
2. 100원딜 내부에서 2인 요금제는 상대적으로 안정적이고, 4인 요금제는 낮게 나오는가?
3. 관측창 내 시청이력 있는 고객과 없는 고객이 모두 보존되었는가?
4. 이후 04번 사용 행동 파생변수 생성에 필요한 `view_history_observation_window.csv`가 생성되었는가?

In [19]:
final_check_100won = membership_preprocessed.groupby("is_100won")["is_repurchase"].agg(
    n="count",
    repurchase_rate="mean",
).reset_index()

display(final_check_100won)

final_check_screen = membership_preprocessed.groupby(["is_100won", "max_screen"], dropna=False)["is_repurchase"].agg(
    n="count",
    repurchase_rate="mean",
).reset_index()

display(final_check_screen)

final_check_watch = membership_preprocessed.groupby(["is_100won", "has_watch_obs"], dropna=False)["is_repurchase"].agg(
    n="count",
    repurchase_rate="mean",
).reset_index()

display(final_check_watch)

final_check_100won.to_csv(TABLES_DIR / "02_preprocessing_final_check_100won.csv", index=False, encoding="utf-8-sig")
final_check_screen.to_csv(TABLES_DIR / "02_preprocessing_final_check_100won_maxscreen.csv", index=False, encoding="utf-8-sig")
final_check_watch.to_csv(TABLES_DIR / "02_preprocessing_final_check_watch_presence.csv", index=False, encoding="utf-8-sig")

,is_100won,n,repurchase_rate
0,0,5939,0.7410
1,1,8983,0.6317


,is_100won,max_screen,n,repurchase_rate
0,0,1,3794,0.7193
1,0,2,1594,0.7842
2,0,4,551,0.7659
3,1,1,5413,0.6580
4,1,2,1528,0.7402
5,1,4,2042,0.4809


,is_100won,has_watch_obs,n,repurchase_rate
0,0,0,1027,0.7459
1,0,1,4912,0.7400
2,1,0,1593,0.6321
3,1,1,7390,0.6317


## 2-17. 02번 노트북 결론

이 노트북에서 확정한 내용은 다음과 같다.

1. 분석 단위는 `membership_row_id`로 추적되는 멤버십 행 또는 구독 이벤트 단위다.
2. `gender == 'N'`, `is_user_verified == 0`, `age == 40` 조합은 더미 이상치로 보고 제외한다.
3. 3주 관측창을 안정적으로 만들기 위해 `subscription_days >= 21`인 행만 유지한다.
4. `price == 100`을 `is_100won`으로 정의하고, `is_promotion`과의 관계를 검산한다.
5. `USER_KEY` 중복 매핑은 오류로 단정하지 않고, 복수 계정 또는 재가입 가능성으로 기록 후 유지한다.
6. 시청이력 관측창은 전체 날짜 기준이 아니라 고객별 `reg_date` 기준 day 0~20으로 정의한다.
7. 4주차 day 21~27은 리텐션 대응기간으로 보고 피처에서 제외한다.
8. 시청이력 없는 고객은 삭제하지 않고, 이후 feature 단계에서 flag로 표현할 수 있도록 보존한다.

다음 노트북은 `03_movie_metadata_unification.ipynb`이다.  
03번에서는 `Movie_Master`를 기준으로 Wavve 크롤링 데이터와 KOBIS 보완 데이터를 통합하고, KOBIS 저신뢰 매칭을 별도 처리한 `movie_metadata_unified_v2.csv`를 생성한다.